# ACMEC 笔记 1:数据、实验过程与当前结论

**研究问题:** Hugging Face 上的开发者在微调前选择父模型(基座),这些选择有多少是因为模型**流行**,有多少是因为模型**适合**(通用质量 + 需求匹配);现状下开发者得到多少效用、损失多少;以后再做反事实(本笔记不涉及)。

- 方法参照 Overgoor, Benson & Ugander,*Choosing to grow a graph*(choose2grow):把每条"子模型 → 父模型"的边看作一次离散选择,用条件 logit / 混合 logit 估计,结果作描述性解读。
- 代码:`/home/caroyf/myproject/ACMEC`;数据与结果:`/net/spaces/jevans/klab/caroyf/data/ACMEC`(仓库内 `data/`、`results/` 是符号链接)。
- 状态截至 2026-11-03。本笔记只作说明,代码单元**不需要运行**。

**目录**
1. 原始数据与关键字段
2. 实验过程(流程图 + 各阶段说明)
3. 当前结论与结果存储位置

## 1. 原始数据与关键字段

### 1.1 来源与规模
| 项目 | 内容 |
|---|---|
| 文件 | `data/raw/ai_ecosystem_withmodelcards.csv`(5.2 GB) |
| 内容 | Hugging Face 模型元数据快照,含完整模型卡(model card)文本 |
| 行数 / 列数 | 1,860,411 个模型 × 23 列(第一列是无名行号) |
| 快照时间 | 最新模型创建于 2025 年中;`downloads` 是快照前 30 天的下载量 |
| 地位 | 主结果只用这份原生数据;外部数据(Open LLM Leaderboard、LMArena)只作稳健性检验 |

### 1.2 字段一览
"非空"按前 2 万行估计。列表类字段在 CSV 中以字符串形式存储(例如 `"['en', 'zh']"`),空列表 `[]` 也算非空。

| 字段 | 含义 | 示例 | 非空 | 本项目中的用途 |
|---|---|---|---|---|
| `model_id` | 仓库 id,`作者/模型名` | `THUDM/GLM-4.1V-9B-Thinking` | 100% | 节点主键;作者取 `/` 前一段 |
| `createdAt` | 创建时间(UTC) | `2025-06-28T14:24:10.000Z` | 100% | 选择发生的时间;2022-03-02 之前的被统一回填,定为 T0 |
| **`parent_model`** | 模型卡声明的全部父模型 | `['THUDM/GLM-4-9B-0414']` | 100% | 父子关系(汇总) |
| **`finetune_parent`** | 全参数微调的父模型 | 同上 | 100% | **主规格的边** |
| **`adapter_parent`** | LoRA 等适配器的父模型 | `[]` | 100% | **主规格的边** |
| `quantized_parent` | 量化版本的来源 | `[]` | 100% | 不算选择;作为"可得性"特征(候选有多少量化版) |
| `merge_parent` | 合并模型的来源 | `[]` | 100% | 不算选择(多父) |
| `pipeline_tag` | 任务标签 | `text-generation` | 69% | 模态划分;需求匹配(任务一致) |
| `library_name` | 框架 | `transformers` | 73% | 可得性特征(工具格式) |
| `tags` | 自由标签 | `['transformers', 'safetensors', ...]` | 100% | 辅助解析库、格式 |
| `languages` | 声明的语言 | `['English', 'Chinese']` | 100% | 需求匹配(语言) |
| `datasets` | 声明的训练数据集 | `[]` | 100% | 需求档案(同数据集的其他模型选了什么) |
| `licenses` | 许可证 | `['apache-2.0']` | 100% | 许可证类别 |
| `likes` | 点赞数(快照时) | `569` | 100% | 结果变量(稳健性) |
| `downloads` | **近 30 天**下载量 | `33839` | 100% | 不用作结果变量(不是累计值) |
| `trendingScore` | HF 热度分 | `367.0` | 100% | 描述用 |
| `arxiv_papers` | 关联论文 | `['2507.01006']` | 100% | 是否有论文 |
| `regions`、`private` | 区域、是否私有 | `['us']`、`False` | 100% | 未使用 |
| **`modelCard`** | 模型卡原文(markdown + YAML) | `---\nlicense: mit\nbase_model: ...` | 66% | model-index 中的 benchmark(P14);卡片文本相似度(P18、P22) |
| `ratelimit_retries`、`exception_raised` | 抓取时的元信息 | `0.0`、空 | — | 未使用 |

### 1.3 使用这些字段时要注意的问题
- **子模型的元数据大量照抄父模型**(P13 继承检验):license、pipeline_tag、languages 的一致率远高于偶然。因此需求**不取**子模型自己的标签,只取:①作者此前选过的父模型(留一处理);②"声明了同一数据集的其他模型"的档案;③子模型卡中开发者自己写的文字(去掉模板、模型名和型号词)。
- **模型卡大多是模板**:HF Trainer 自动生成的段落,而且直接写出父模型名。去掉这些之后,只有约 26% 的作者首次选择还剩开发者自己写的内容。
- **`createdAt` 回填**:2022-03-02 之前的模型都是同一时刻,这些边只作为初始状态。
- **被删除的父模型**:例如 runwayml/stable-diffusion-v1-5 有 3,190 个子模型。子模型 ≥ 100 的悬空父模型补为占位节点。
- **批量账号**:410 个账号贡献了 52% 的选择。主结果只计每个"作者–父模型"对的第一次选择(方案 A)。

### 1.4 由原始数据构建的分析数据
| 数据 | 路径(相对 `data/`) | 规模 |
|---|---|---|
| 节点表 | `processed/nodes/nodes.parquet` | 1,860,413 个模型(含 2 个占位节点) |
| 边表 | `processed/edges/edges.parquet` | finetune 188,819 / adapter 243,643 / quantized 101,095 / merge 40,220 条 |
| 主规格选择 | `processed/choices/main_strat/` | 422,714 次选择(finetune + adapter + unspecified,T0 之后,同模态);作者首次选择 133,390 次 |
| 建模样本 | `processed/choices/main_strat_frac0.1/` | 10% 抽样:42,265 次选择、1,344 万候选行;方案 A 下 13,341 次 |
| 候选集 | 同上 `rows/` | 每次选择约 318 个候选:500 个热门父模型(全部)+ 已有子模型的模型(抽样)+ 零子模型的模型(抽样),带 log q 修正 |
| 特征 | `p15_rows.parquet`、`match.parquet`、`text.parquet`、`emb.parquet` | 每个"选择 × 候选"一行 |

In [ ]:
# 仅作示例,不需要运行(原始 CSV 很大,读模型卡很慢)
import pandas as pd
cols = ["model_id", "createdAt", "pipeline_tag", "library_name", "languages", "datasets",
        "finetune_parent", "adapter_parent", "quantized_parent", "merge_parent", "likes", "downloads"]
raw = pd.read_csv("/net/spaces/jevans/klab/caroyf/data/ACMEC/raw/ai_ecosystem_withmodelcards.csv",
                  usecols=cols, nrows=1000, dtype=str)
nodes = pd.read_parquet("/net/spaces/jevans/klab/caroyf/data/ACMEC/processed/nodes/nodes.parquet")
edges = pd.read_parquet("/net/spaces/jevans/klab/caroyf/data/ACMEC/processed/edges/edges.parquet")

## 2. 实验过程

### 2.1 流程图
(Mermaid 图;若当前编辑器不渲染,见下方文字版。)

```mermaid
flowchart TD
    A[原始 CSV<br/>1,860,411 个模型] --> B[P1–P2 载入、审计、清洗<br/>节点表 / 边表]
    B --> C[P3 描述统计<br/>集中度、degree 分布]
    B --> D[P4 时间重放 + 分层负采样<br/>每次选择约 318 个候选,log q 修正]
    D --> E[P5–P9 条件 logit 阶梯、degree 曲线<br/>惯性、异质性、EM 混合模型]
    E -. 旧主线,降为附录 .-> X[P0b–P0f、GMM、IIA、P10–P12<br/>因果性优先连接 / 锁定]

    D --> F[P13 继承检验 + 需求匹配特征]
    A --> G[P14 model-index benchmark]
    A --> H[P18 模型卡文本 TF-IDF]
    F --> I
    G --> I
    H --> I
    I[P15 分解:七组特征<br/>Shapley / latent class / 边际替代率 / 逐次分类]
    I --> J[P16 福利:logsum、适合损失<br/>按 6 个月内子模型数定标]
    I --> K[P17 同权重配对<br/>收紧质量上界]

    I --> R{P19–P23 加固}
    R --> R1[P19 按作者 bootstrap<br/>13 种设定]
    R --> R2[P20 安慰剂 + 样本外]
    R --> R3[P21 外部质量数据<br/>Leaderboard / Arena]
    R --> R4[P22 文本嵌入 + LLM 标注]
    R --> R5[P23 考虑集模型 + 嵌套 logit]
    R1 --> Z[当前结论]
    R2 --> Z
    R3 --> Z
    R4 --> Z
    R5 --> Z
    J --> Z
    K --> Z
    Z -. 下一阶段,未开始 .-> CF[反事实分析]
```

**文字版**
```
原始 CSV ─► P1–P2 清洗(节点/边) ─► P4 重放 + 负采样 ─► P5–P9 基础模型
                                           │
              P13 需求特征 ─┐              │
              P14 benchmark ─┼──────────► P15 分解(流行 / 可见度 / 可得性 / 质量 / 匹配 / 惯性 / 局部搜索)
              P18 卡片文本 ─┘                     │
                                                  ├─► P16 福利     ├─► P17 收紧质量上界
                                                  └─► P19–P23 加固(bootstrap、安慰剂、外部质量、嵌入/LLM、考虑集)
                                                                     └─► 当前结论 ─ ─► 反事实(未开始)
```

### 2.2 核心设定
- **一次选择** = 一个子模型在出生时刻从当时已存在、同模态的模型中选一个父模型。
- **候选集**:分层负采样,效用里减去 log q,做法与论文第 3 节一致。
- **主样本(方案 A)**:只计每个"作者–父模型"对的第一次选择,并从候选集中剔除作者以前用过的父模型。用全部选择、按 n^-0.5 降权的版本作对照。
- **七组特征**:

| 组 | 含义 | 特征 |
|---|---|---|
| 流行 | 对所有人相同的社会信号 | 累积 degree、近 30 天新增子模型、大机构 |
| 可见度 / 生命周期 | 是否被用过、发布多久 | has_deg、legacy、年龄 |
| 可得性 | 好不好拿来用 | 量化版本数、工具格式 |
| 通用质量 | 对所有人都更好 | model-index benchmark;上界用热门父模型的固定效应(FE) |
| 需求匹配 | 因人而异 | 规模、任务、语言、数据集档案、卡片文本相似度 |
| 惯性 | 习惯与照抄 | 同家族、自有模型、上一次的父模型 |
| 局部搜索 | 沿已有关系寻找 | 两跳共用、同数据集共用 |

- **份额口径**:主要用对数似然提升的 Shapley 分解(2^7 次拟合);另报告 latent class 模式占比、边际替代率、逐次选择分类。
- **"适合 / 流行"比值** = (质量 + 匹配) 的份额 ÷ 流行的份额;可见度、可得性单列。

### 2.3 各阶段说明(新研究方向下)
| 阶段 | 问题 | 做法 | 脚本 |
|---|---|---|---|
| P13 | 子模型标签能否代表需求? | 照抄率与偶然一致率比较;构造需求匹配特征 | `scripts/p13_inherit.py`、`p13_match.py`、`src/hfpa/features/match.py` |
| P14 | 能否直接测量质量? | 解析模型卡 model-index 中的 Open LLM Leaderboard 分数 | `scripts/p14_quality.py` |
| P15 | 流行和适合各占多少? | 条件 logit + Shapley、latent class(半合成验证)、边际替代率、逐次分类 | `scripts/p15_decomp.py` |
| P16 | 现状效用和损失多少? | logsum;适合损失;用 6 个月内子模型数定标(带和不带作者 FE) | `scripts/p16_welfare.py`、`p16_calib_full.py` |
| P17 | 热门父模型 FE 中有多少不是质量? | 15 对权重完全相同的副本 | `scripts/p17_mirror_quality.py` |
| P18 | 模型卡文本能否补充需求? | 去模板、去名称后算 TF-IDF 相似度 | `scripts/p18_text.py`、`src/hfpa/features/text.py` |
| P19 | 数字有多稳? | 按作者 bootstrap(无 FE 197 次、有 FE 40 次);13 种设定 | `scripts/p19_robust.py` |
| P20 | 匹配特征是否真反映个人需求? | 换成同月同模态另一位作者的需求(安慰剂);按时间外推的样本外检验 | `scripts/p20_placebo.py` |
| P21 | 外部质量能解释多少 FE? | Open LLM Leaderboard v2、LMArena;同家族规模阶梯;经验作者 vs 其他作者 | `scripts/p21_quality_ext.py` |
| P22 | 更好的文本信号;LLM 能否判断"对口"? | e5 嵌入;Qwen2.5-14B / Llama-3.1-8B 两两对比标注 300 次选择 | `scripts/p22_embed.py`(借用 ICML 的环境和模型,只读) |
| P23 | 流行通过"被看见"还是"效用"起作用? | 考虑集模型(注意力 × logit)、按家族的嵌套 logit,附识别检验 | `scripts/p23_consider.py`、`src/hfpa/models/consider.py` |

旧的因果主线(P0b–P0f、P5 GMM、P9 IIA、P10–P12)保留为附录,见 README 末尾。

## 3. 当前结论与结果存储位置

### 3.1 当前结论(方案 A,13,341 次作者首次选择)

**① 大多数选择跟着热度走,适合只占一小部分。**

| 口径 | 适合份额 | 适合 / 流行 |
|---|---|---|
| 下界(质量只用 benchmark),bootstrap 95% CI | 5.4%(5.1–5.7%) | 0.127(0.119–0.134) |
| 去掉安慰剂也能得到的部分(只算个人需求) | 约 2.8% | 约 0.06 |
| 只把外部质量能解释的 FE 算作质量(P21) | 5–11% | 0.09–0.21 |
| P17 收紧后的上界 | 22.0%(21.7–22.3%) | 0.56 |
| 原始上界(FE 全算质量) | 34.2% | 1.26 |

- **当前最可信的区间:适合份额约 3–11%,比值约 0.06–0.2。** 原来的上界明显偏高,因为热门父模型的 FE 主要不是质量。
- 流行(28–43%)加可见度(20–27%)在任何口径下都是主要驱动;可得性约 12–16%。
- 13 种设定下比值大多在 0.08–0.18;图像生成模型例外(0.012,几乎全靠流行和可得性)。

**② 需求匹配是真信号,但覆盖有限。**
- 候选与作者以往任务一致,相当于 degree 翻 243 倍;换成别人的需求后,这个系数从 1.91 降到 0.08。
- 原匹配份额 5.4% 中,约一半(语言匹配、候选有没有卡片等)不是个人需求,而是候选自身的属性。
- 样本外仍有增益,但只占总提升的 0.8–2.6%。
- 约三分之一的选择看不出任何需求线索;e5 嵌入没有比 TF-IDF 更好。

**③ 热门父模型的"固定效应"主要不是质量(P17、P21)。**
- unsloth 的副本比权重完全相同的原模型 FE 高 1.84,这个发布者溢价占 FE 方差的 42%。
- 外部 benchmark 和 Arena 评分与 FE 呈负相关;同家族里参数越大 FE 越低(Spearman −0.53)。FE 更像"副本 / 免门控 / 小模型好用"。

**④ 流行部分通过"被看见"起作用(P23)。**
- "流行只决定被看见"的模型拟合明显变差;"既影响被看见、也影响效用"的模型拟合最好。
- 累积 degree 和大机构品牌主要影响"被看见",近期热度两种渠道都有;适合特征的系数基本不变。
- 没有发现同家族内的额外替代(嵌套 logit 的 λ = 1)。

**⑤ 福利:个人损失很小,代价在生态层面。**
- 选得不够匹配,在同一作者内部只造成约 0.2% 的结果损失(6 个月内子模型数),不控制作者约 1.4%。
- 流行型选择:Top-10 份额 91%,有效父模型数约 9;适合型选择:12%,约 554;整体:22%,839。

**⑥ LLM 标注(P22)暂不可用。**
- Llama-3.1-8B 有严重的位置偏差(300 条里 254 条选 A)。
- Qwen2.5-14B 没有位置偏差,能判断的情况下 83% 认为被选父模型更对口。但提示里给了子模型的任务标签,而它 69% 照抄父模型,很可能高估。
- 等用户抽查 50 条,并去掉任务标签后重标。

### 3.2 结果存储位置
根目录:`/net/spaces/jevans/klab/caroyf/data/ACMEC`(仓库内通过 `data/`、`results/` 访问)。

| 内容 | 位置 |
|---|---|
| 结果表(每张都有 `.md` 和 `.csv`) | `results/tables/` |
| 分解(P15) | `p15_shapley*`、`p15_mixture_*`、`p15_mrs`、`p15_classify`、`p15_clogit*` |
| 福利(P16) | `p16_welfare_*`、`p16_calib_full*` |
| 质量上界(P17) | `p17_mirror_quality`、`p17_mirror_pairs_fe` |
| 文本(P18) | `p18_text`;加入文本前的旧结果在 `results/tables/pre_p18/` |
| bootstrap 与设定矩阵(P19) | `p19_boot`、`p19_classify_boot`、`p19_spec`;逐次结果在 `results/tables/p19/`;图 `results/figures/p19_ratio_forest.png` |
| 安慰剂与样本外(P20) | `p20_placebo_coef`、`p20_placebo_desc`、`p20_placebo_shapley`、`p20_oot`、`p20_oot_fit` |
| 外部质量(P21) | `p21_quality_ext`、`p21_implied`、`p21_size_ladder*`、`p21_expert`、`p21_hubs_quality.csv` |
| 嵌入与 LLM 标注(P22) | `p22_emb_*`、`p22_labels`;**待用户填写的抽查表 `p22_audit.csv`**;LLM 原始输出 `/net/spaces/jevans/klab/caroyf/outputs/p22/` |
| 考虑集与嵌套 logit(P23) | `p23_consider`、`p23_consider_coef`、`p23_sim` |
| 图 | `results/figures/` |
| 阶段报告与快照(入库) | `reports/2026-10-29`、`2026-10-30`、`2026-10-31`(每个含报告、`tables/`、`figures/`);P19–P23 尚未生成快照 |
| 中间数据 | `interim/`(`cards_clean.parquet`、`cards_masked.parquet`、`text_tfidf.npz`、`cards_emb.npy` 等) |
| 外部数据(仅稳健性) | `external/quality/`(Leaderboard v2、LMArena);嵌入模型 `external/models/multilingual-e5-base` |
| 借用的资源(只读) | ICML 项目的 `qc-vllm` 环境和 `/net/spaces/jevans/klab/caroyf/data/ICML/models/hf/` 下的模型,见 README "借用的外部资源" |
| 运行日志 | `/net/spaces/jevans/klab/caroyf/logs/acmec_{cpu,gpu,vllm}_<作业号>.out` |
| 总说明 | 仓库 `README.md`(路线图、数据问题、结果);阶段报告 `results/report_2026-10-29.md` |

### 3.3 尚未完成
1. P22:用户抽查 50 条 LLM 标注(`results/tables/p22_audit.csv`);去掉任务标签后重标。
2. P19–P23 写进 README 和报告第 9 节,生成新快照,然后提交;推送需用户同意。
3. 反事实分析尚未开始。P23 表明,"让开发者看到更多候选"可以作为反事实的一种设定。